In [1]:
import json
import pandas as pd


In [2]:
path = '../data/characters_detail_parallel.json'
with open(path, 'r') as file:
    # Load the JSON data from the file
    data = json.load(file)


In [4]:
new_data = []
for d in data:
    keys = d.keys()
    new_id = list(keys)[0]
    detail = d[new_id]
    detail['id'] = new_id
    new_data.append(detail)
df = pd.DataFrame(new_data)
print(df)


           jname           rname  \
0        [シャンクス]      [Shankusu]   
1     [ベン・ベックマン]  [Ben Bekkuman]   
2      [ラッキー・ルウ]      [Rakkī Rū]   
3         [ヤソップ]       [Yasoppu]   
4      [ライムジュース]     [Raimujūsu]   
...          ...             ...   
1368         NaN             NaN   
1369  [ピープリー･ルル]   [Pīpurī Ruru]   
1370      [クラップ]       [Kurappu]   
1371       [ジニー]          [Jinī]   
1372       [ベコリ]        [Bekori]   

                                               ename  \
0                                           [Shanks]   
1        [Benn Beckman, Ben Beckman (VIZ, formerly)]   
2                    [Lucky Roux, Lucky Roo (WT100)]   
3                                           [Yasopp]   
4                                       [Lime Juice]   
...                                              ...   
1368                                             NaN   
1369  [Peepley Lulu (Viz), Peeply Lulu (Funimation)]   
1370                                         [Clapp]   
1371   

In [25]:
from IPython.display import display, HTML
# Assuming df is your DataFrame
non_nan_counts = df.count().sort_values(ascending=False)
non_nan_counts_df = non_nan_counts.to_frame(name='available')
# print(non_nan_counts_df)
max_value = non_nan_counts_df['available'].max()
non_nan_counts_df['percentage'] = ((non_nan_counts_df['available'] / max_value) * 100).round(2)

# # Split the Series into two parts
split_index = len(non_nan_counts_df) // 2
first_part = non_nan_counts_df[:split_index]
second_part = non_nan_counts_df[split_index:]

# # # Display the two parts as separate HTML tables
display(HTML(first_part.to_html()))
display(HTML(second_part.to_html()))


,available,percentage
id,1373,100.00
status,1290,93.95
rname,1290,93.95
jname,1290,93.95
ename,1290,93.95
first,1290,93.95
occupation,1128,82.16
affiliation,1127,82.08
jva,991,72.18
Funi eva,941,68.54


,available,percentage
bounty,183,13.33
dfname,163,11.87
dfmeaning,161,11.73
4kids eva,108,7.87
alias,86,6.26
Odex eva,69,5.03
liveaction,50,3.64
age2,44,3.20
doriki,8,0.58
zombie number,8,0.58


In [26]:
def extract_single_element(lst):
    if not isinstance(lst, list):
        return lst
    if len(lst) == 0:
        return None
    return lst[0] if len(lst) == 1 else lst


In [31]:
def extract_single_element(lst):
    if not isinstance(lst, list):
        return lst
    if len(lst) == 0:
        return None
    return lst[0] if len(lst) == 1 else lst

df = df.applymap(extract_single_element)


/tmp/ipykernel_435234/85927778.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(extract_single_element)


In [37]:
def get_region(origin):
    regions = ['East Blue', 'West Blue', 'North Blue', 'South Blue', 'Grand Line', 'Sky']
    if not origin:
        return None
    if not isinstance(origin, str):
        return origin
    for r in regions:
        if r in origin:
            return r
    return origin


In [38]:
regions = df['origin'].apply(get_region)
df['region'] = df['origin'].apply(get_region)
count_per_value = regions.value_counts().reset_index()

# Rename the columns for clarity
count_per_value.columns = ['region', 'count']

display(HTML(count_per_value.to_html()))


,region,count
0,Grand Line,216
1,East Blue,112
2,North Blue,50
3,West Blue,32
4,South Blue,30
5,Calm Belt (Amazon Lily),22
6,Sky,17
7,Wano Country,14
8,Red Line (Mary Geoise),5
9,Jaya,2


In [135]:
def get_first_chapter(first_appeareances):
    chapter_string = ''
    try:
        # if pd.isna(first_appeareances):
        #     return None
        if not first_appeareances:
            return None
        if not isinstance(first_appeareances, list):
            chapter_string = first_appeareances
        for appear in first_appeareances:
            if 'chapter' in appear.lower():
                chapter_string = appear
                break
        if not chapter_string:
            return None
        else:
            chapter = chapter_string.split(' ')
            # chapter.replace(',', '')
            return int(chapter[1].replace(',', ''))
    except ValueError:
        print(f'error: {first_appeareances}')
        print(f'error: {chapter[1]}')
        return None
    except Exception as e:
        return None


In [136]:
first_chapters = df['first'].apply(get_first_chapter)
print(f'Before time skip: {(first_chapters <= 600).sum()}')
print(f'After time skip: {(first_chapters > 600).sum()}')


Before time skip: 667
After time skip: 622


In [153]:
def get_age(ages):
    age_string = ''
    try:
        # if pd.isna(first_appeareances):
        #     return None
        if not ages:
            return None
        if not isinstance(ages, list):
            age_string = ages

        age_strings = [a.split(' ')[0] for a in ages]
        # print(age_strings)
        return(int(age_strings[-1]))
    except ValueError:
        return None
    except Exception as e:
        return None


In [250]:
# df['age'][0][0]
df['true_age'] = df['age'].apply(get_age)

print(df[['ename', 'true_age']].sort_values('true_age', ascending=False).head(20))


                                                 ename  true_age
145                                              Dorry       160
143                             [Broggy, Brogy (Odex)]       160
405                                             Kashii       156
404                                               Oimo       153
157                                             Kureha       141
424                                     Jaguar D. Saul       127
669                                      San Juan Wolf        99
558                                            Haredas        97
451                                              Brook        90
274                                            Sengoku        79
281                                             Amazon        79
48                                    Silvers Rayleigh        78
108                                     Monkey D. Garp        78
275                                              Tsuru        76
44                       

In [201]:
def get_height(heights):
    age_string = ''
    # print(heights)
    try:
        # if pd.isna(first_appeareances):
        #     return None
        if not heights:
            return None
        if not isinstance(heights, list):
            heights = [heights]
        # print(f'unsplitted age: {age_strings}')

        age_strings = [a.split(' ')[0] for a in heights]
        # print(f'one age: {age_strings}')
        if 'cm' in heights[0]:
            return(int(age_strings[0]))
        elif 'km' in heights[0]:
            return(int(age_strings[0]) * 100000)
        else:
            return(int(age_strings[0]) * 100)
    except ValueError:
        return None
    except Exception as e:
        return None


In [248]:
df['height'][0]
df['true_height'] = df['height'].apply(get_height)
print(df[['ename', 'true_height']].sort_values('true_height', ascending=False).head(20))


                                              ename  true_height
126                         [Laboon, Raboon (Odex)]       40,000
669                                   San Juan Wolf       18,000
717                                       Wadatsumi        8,000
477                                            Oars        6,700
1281                                         Haccha        6,680
646                                 Little Oars Jr.        6,000
793                              Yeti Cool Brothers        4,250
101   [Momoo (4Kids, Viz, FUNimation), Morm (Odex)]        3,600
983                                        Kingbaum        2,340
145                                           Dorry        2,260
851                                        Hajrudin        2,200
1050                                          Jorul        2,150
143                          [Broggy, Brogy (Odex)]        2,130
1051                                          Jarul        2,050
424                      

In [247]:
def get_bounty(heights):
    age_string = ''
    # print(heights)
    try:
        # if pd.isna(first_appeareances):
        #     return None
        if not heights:
            return None
        if not isinstance(heights, list):
            heights = [heights]
        max_bounty =  max([int(h) for h in heights])
        if max_bounty > 10e+10:
            return max_bounty / 10e+7
        else:
            return max_bounty
    except ValueError:
        return None
    except Exception as e:
        return None
# df['bounty'][0]
pd.set_option('display.float_format', lambda x: '{:,.0f}'.format(x))
df['true_bounty'] = df['bounty'].apply(get_bounty)
print(df[['ename', 'true_bounty']].sort_values('true_bounty', ascending=False).head(25))


                                                  ename   true_bounty
8                                          Gol D. Roger 5,564,800,000
195   [Edward Newgate (VIZ, Funimation), Ward Newgat... 5,046,000,000
438                                               Kaido 4,611,100,000
437                                    Charlotte Linlin 4,388,000,000
0                                                Shanks 4,048,900,000
158   [Marshall D. Teech (VIZ, Funimation subs, edit... 3,996,000,000
84                                       Dracule Mihawk 3,590,000,000
34                                                Buggy 3,189,000,000
9                                       Monkey D. Luffy 3,000,000,000
505                              Trafalgar D. Water Law 3,000,000,000
501                                         Eustass Kid 3,000,000,000
1327                                        Adio Suerte 2,009,430,000
140                                           Crocodile 1,965,000,000
541                 

In [213]:
df['bounty'][0]


['4048900000', '1040000000']

In [249]:
df['region'] = df['origin'].apply(get_region)
pd.set_option('display.float_format', lambda x: '{:,.0f}'.format(x))
result = df.groupby('region')['true_bounty'].sum()
print(result.sort_values(ascending=False))


region
Grand Line                41,271,291,000
East Blue                 11,951,300,000
West Blue                  7,423,400,000
North Blue                 5,476,000,000
South Blue                 4,625,000,000
Calm Belt (Amazon Lily)    1,739,000,000
Wano Country                 480,000,000
Red Line (Mary Geoise)       340,000,000
Sky                          108,000,000
Zou                                  500
Jaya                                   0
Lvneel Kingdom                         0
New World (Waford)                     0
Punk Hazard                            0
Sorbet Kingdom                         0
Name: true_bounty, dtype: float64


In [254]:
non_nan_values = df['dftype'].dropna().value_counts()
print(non_nan_values)


dftype
Paramecia                       100
Artificial Zoan                  28
Zoan                             21
Logia                            15
Ancient Zoan                      9
Mythical Zoan                     8
Artificial Paramecia              4
[Paramecia, (Mythical Zoan)]      1
Cow Zoan                          1
Koala Zoan                        1
Zebra Zoan                        1
Rhinoceros Zoan                   1
Chihuahua Zoan                    1
Artificial Mythical Zoan          1
Special Paramecia                 1
Unknown                           1
Name: count, dtype: int64


In [265]:
def get_generic_df_type(df_type):
    try:
        df_types = ['Paramecia', 'Zoan', 'Logia']
        if not df_type:
            return None
        if isinstance(df_type, list):
            return df_type
        for r in df_types:
            if r in df_type:
                return r
        return df_type
    except:
        return df_type


In [266]:
df['generic_df_type'] = df['dftype'].apply(get_generic_df_type)
count_per_value = df['generic_df_type'].value_counts().reset_index()

# Rename the columns for clarity
count_per_value.columns = ['DF Type', 'count']

display(HTML(count_per_value.to_html()))


,DF Type,count
0,Paramecia,105
1,Zoan,72
2,Logia,15
3,"[Paramecia, (Mythical Zoan)]",1
4,Unknown,1


In [268]:
df['status'].value_counts().reset_index()


,status,count
0,Alive,1038
1,Unknown,135
2,Deceased,117


In [274]:
df['blood type'].value_counts().reset_index()


,blood type,count
0,F,113
1,S,105
2,X,101
3,XF,63
4,S (RH-),9
5,Green Blood,4
6,"[Bas: X, And: XF, Kerville: F]",1


In [285]:
df[df['blood type'] == 'S (RH-)']['ename']


75                                                 Sanji
553                                         Fisher Tiger
744    [Splash and Splatta, Splash and Splatter (WT100)]
793                                   Yeti Cool Brothers
824                                            Dellinger
953                                       Vinsmoke Yonji
962                                      Vinsmoke Ichiji
963                                        Vinsmoke Niji
973                                    Charlotte Praline
Name: ename, dtype: object

In [293]:
df['birth'].value_counts()


birth
September 29th    6
February 28th     5
May 19th          5
February 4th      5
July 18th         5
                 ..
May 3rd           1
November 12th     1
May 31st          1
November 10th     1
March 15th        1
Name: count, Length: 374, dtype: int64

In [296]:
from datetime import datetime
import re

# Function to convert a date string with ordinal suffix to datetime
def parse_date_with_suffix(date_str):
    try:
        # Remove the ordinal suffix (st, nd, rd, th) from the day
        date_str = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_str)

        # Convert the date string to a datetime object
        dt_object = datetime.strptime(date_str, '%B %d')

        return dt_object
    except:
        return None

parse_date_with_suffix(df['birth'][0])


datetime.datetime(1900, 3, 9, 0, 0)

In [300]:
df['birth_date'] = df['birth'].apply(parse_date_with_suffix)
df['birth_date']

grouped_by_month = df.groupby(df['birth_date'].dt.month)['birth_date'].count()
print(grouped_by_month)


birth_date
1     54
2     63
3     59
4     50
5     52
6     51
7     52
8     61
9     67
10    68
11    55
12    54
Name: birth_date, dtype: int64


In [336]:
current_date = datetime.now().date()

# Filter rows where 'date' is equal to the current date
selected_rows = df[(df['birth_date'].dt.month == current_date.month) & (df['birth_date'].dt.day == current_date.day)]
print(selected_rows['id'])


544    Ran
Name: id, dtype: object


In [332]:
for k, v in enumerate(df.columns.tolist()):
    print(k, v)


0 jname
1 rname
2 ename
3 first
4 affiliation
5 occupation
6 origin
7 epithet
8 status
9 age
10 birth
11 height
12 blood type
13 bounty
14 jva
15 Odex eva
16 4kids eva
17 Funi eva
18 liveaction
19 id
20 residence
21 alias
22 age2
23 dfname
24 dfename
25 dfmeaning
26 dftype
27 birthname
28 dfname2
29 dfename2
30 dfmeaning2
31 dftype2
32 size
33 weight
34 doriki
35 cp9key
36 zombie number
37 gladiator number
38 true_age
39 true_height
40 true_bounty
41 true_bounty_normal
42 region
43 generic_df_type
44 birth_date


In [334]:
current_date = datetime.now().date()
current_date.month


11